# 06. 속성별 지식 검색 및 HS 후보 재정렬

05번은 제품 전체 텍스트를 하나의 검색 질의로 사용했습니다. 06번에서는 제품명·성분·형태·용도를 분리해 각각 검색하고, 속성별 검색 점수를 결합해 후보 코드의 근거 점수를 계산합니다.

목표는 검색 근거의 적합성을 높이는 것입니다. 이 노트북은 정답 코드를 검색이나 재정렬에 사용하지 않으며, 정답은 마지막 평가에만 사용합니다. 가중치별 결과는 탐색적 비교로 해석하고, 같은 CV 결과를 반복 조정해 최종 성능을 주장하지 않습니다.


In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv', safe='')
data_all = pd.read_csv(BASE + cases_file).fillna('')
kb = pd.read_json(BASE + 'hs_knowledge_base.jsonl', lines=True).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
field_weights = {'Product Name':0.30,'Main Ingredients (KR)':0.20,'Main Ingredients (EN)':0.10,'Product Form and Packaging':0.20,'Intended Use':0.20}
data_all['model_text'] = data_all[input_cols].astype(str).agg(' '.join, axis=1)
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join, axis=1).map(lambda x: hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby('HS Code').filter(lambda x: len(x) >= 4)
group_counts = eligible.groupby('HS Code')['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts >= 4].index.astype(str))
data = eligible[eligible['HS Code'].astype(str).isin(labels)].copy().reset_index(drop=True)
data['HS Code'] = data['HS Code'].astype(str)
print('제품:',len(data),'| HS 코드:',data['HS Code'].nunique(),'| KB 문서:',len(kb))


## 1. 지식 문서 유형 정리

제외 여부는 문서 본문의 단순 키워드가 아니라 `rule_type`의 명시적 값으로 판별합니다. `소호별 포함 범위 및 제외`처럼 포함과 제외가 함께 있는 문서는 자동 긍정 근거에서 제외하고 별도 혼합 규칙으로 보존합니다. `조건부 포함`은 조건 확인이 필요하므로 긍정 근거로 자동 가산하지 않습니다.


In [ ]:
def code_digits(value):
    return re.sub(r'\\D','',str(value))

def code_compatible(rule_code, candidate_code):
    a,b = code_digits(rule_code),code_digits(candidate_code)
    return bool(a and b and (a.startswith(b) or b.startswith(a)))

kb['rule_type_clean'] = kb['rule_type'].astype(str).str.strip()
kb['is_exclusion'] = kb['rule_type_clean'].isin(['제외','제외·예외'])
kb['is_mixed'] = kb['rule_type_clean'].eq('소호별 포함 범위 및 제외')
positive_types = {'포함','소호별 포함 범위','분류원칙·예외'}
kb['is_positive'] = kb['rule_type_clean'].isin(positive_types)
kb['code_digits'] = kb['hs_code'].map(code_digits)
print('긍정 규칙:',int(kb['is_positive'].sum()),'| 제외 규칙:',int(kb['is_exclusion'].sum()),'| 혼합 규칙:',int(kb['is_mixed'].sum()))
display(kb.groupby(['source_sheet','rule_type_clean']).size().rename('count').reset_index())


## 2. 속성별 TF-IDF 검색기

각 제품 속성을 별도의 질의로 검색합니다. 각 속성별로 문서와의 cosine similarity를 계산하고, 후보 코드와 계층 호환되는 긍정 규칙 중 가장 높은 점수를 속성별 근거 점수로 사용합니다.


In [ ]:
kb['search_text'] = (kb['document_text'].astype(str)+' '+kb['rule_type_clean'].astype(str)+' '+kb['source_sheet'].astype(str)).str.strip()
kb_vectorizer = FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),max_features=30000,sublinear_tf=True))])
kb_matrix = kb_vectorizer.fit_transform(kb['search_text'])
def field_similarity(value):
    value = str(value).strip()
    if not value: return np.zeros(len(kb))
    q = kb_vectorizer.transform([value])
    return (kb_matrix @ q.T).toarray().ravel()


## 3. 속성별 근거 점수 및 후보 재정렬

각 속성별로 후보 코드와 호환되는 명시적 긍정 규칙의 최대 유사도를 계산합니다. 속성 가중치는 초기 가설이며, 정답에 맞춰 조정하지 않습니다. 제외·혼합·조건부 규칙은 자동 긍정 점수에 포함하지 않고 별도 evidence로 저장합니다.


In [ ]:
ALPHAS = [0.10,0.20,0.30]
N_SPLITS = 4
cv = StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)
oof_rows, evidence_rows, audit_rows = [],[],[]
for fold,(tr,te) in enumerate(cv.split(data['model_text'],data['HS Code'],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr],data.iloc[te]
    vec = FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))])
    Xtr,Xte = vec.fit_transform(train['model_text']),vec.transform(test['model_text'])
    clf = LinearSVC(C=1,class_weight='balanced',max_iter=5000)
    clf.fit(Xtr,train['HS Code'])
    decision = clf.decision_function(Xte)
    if decision.ndim == 1: decision = np.column_stack([-decision,decision])
    for local_i,(_,row) in enumerate(test.iterrows()):
        order = np.argsort(decision[local_i])[::-1][:3]
        candidates = [(str(clf.classes_[j]),float(decision[local_i,j])) for j in order]
        scores_by_field = {field:field_similarity(row[field]) for field in field_weights}
        field_candidate_scores = {}
        for code,svm_score in candidates:
            compatible = kb['hs_code'].map(lambda x:code_compatible(x,code))
            pos_mask = kb['is_positive'] & compatible
            attr_scores = {}
            for field,weight in field_weights.items():
                vals = scores_by_field[field][pos_mask.to_numpy()]
                attr_scores[field] = float(vals.max()) if len(vals) else 0.0
            evidence_score = sum(field_weights[f]*attr_scores[f] for f in field_weights)
            field_candidate_scores[code] = {'svm_score':svm_score,'evidence_score':evidence_score,'attribute_scores':attr_scores}
            for field,weight in field_weights.items():
                sims = scores_by_field[field]
                relevant_idx = np.argsort(sims)[::-1][:10]
                for kb_idx in relevant_idx:
                    d = kb.iloc[kb_idx]
                    if not (bool(d['is_exclusion']) or bool(d['is_mixed']) or bool(d['is_positive']) or str(d['rule_type_clean']) in {'조건부 포함','조건부 분류'}): continue
                    if not code_compatible(d['hs_code'],code) and str(d['hs_code']).strip(): continue
                    evidence_rows.append({'fold':fold,'row_index':int(row.name),'candidate_hs':code,'field':field,'field_weight':weight,'knowledge_id':d['knowledge_id'],'source_sheet':d['source_sheet'],'rule_hs_code':d['hs_code'],'rule_type':d['rule_type_clean'],'evidence_category':'exclusion' if d['is_exclusion'] else 'mixed' if d['is_mixed'] else 'positive' if d['is_positive'] else 'conditional','similarity':float(sims[kb_idx]),'document_text':d['document_text']})
        svm_vals = np.array([field_candidate_scores[c[0]]['svm_score'] for c in candidates],dtype=float)
        svm_min,svm_max = svm_vals.min(),svm_vals.max()
        svm_norm = (svm_vals-svm_min)/(svm_max-svm_min) if svm_max>svm_min else np.ones(len(svm_vals))
        for alpha in ALPHAS:
            combined = {candidates[j][0]:float(svm_norm[j]+alpha*field_candidate_scores[candidates[j][0]]['evidence_score']) for j in range(len(candidates))}
            ranked = sorted(combined,key=combined.get,reverse=True)
            oof_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row['HS Code'],'alpha':alpha,'svm_top1':candidates[0][0],'svm_top3':' | '.join(c[0] for c in candidates),'reranked_top1':'','reranked_top3':' | '.join(ranked),'candidate_scores':str(field_candidate_scores),'combined_scores':str(combined)})
            oof_rows[-1]['reranked_top1'] = ranked[0]
    audit_rows.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':len(set(train['duplicate_group'])&set(test['duplicate_group']))})
    print('fold',fold,'완료')

oof = pd.DataFrame(oof_rows)
evidence = pd.DataFrame(evidence_rows)
fold_audit = pd.DataFrame(audit_rows)
print('OOF rows:',len(oof),'| evidence rows:',len(evidence))
display(fold_audit)


## 4. 성능 비교 및 alpha 민감도

SVM 기준선은 각 제품의 `svm_top1`을 사용합니다. 재정렬 결과는 alpha별로 비교합니다. Top-3 후보 집합은 동일하므로 Top-3 hit 여부도 함께 확인합니다.


In [ ]:
def score_metrics(frame,pred_col):
    return {'accuracy':accuracy_score(frame['true_hs'],frame[pred_col]),'macro_f1':f1_score(frame['true_hs'],frame[pred_col],average='macro',zero_division=0),'weighted_f1':f1_score(frame['true_hs'],frame[pred_col],average='weighted',zero_division=0)}

metric_rows = []
for alpha in ALPHAS:
    sub = oof[oof['alpha']==alpha].copy()
    metric_rows.append({'method':'SVM baseline','alpha':0.0,**score_metrics(sub,'svm_top1')})
    metric_rows.append({'method':'Attribute KB reranking','alpha':alpha,**score_metrics(sub,'reranked_top1')})
metrics = pd.DataFrame(metric_rows).drop_duplicates(subset=['method','alpha'])
top3 = oof[oof['alpha']==ALPHAS[0]].copy()
top3['top3_hit'] = top3.apply(lambda r:r['true_hs'] in r['svm_top3'].split(' | '),axis=1)
print('Top-3 hit rate:',top3['top3_hit'].mean())
display(metrics)
change_summary = oof.groupby('alpha').apply(lambda x:(x['svm_top1']!=x['reranked_top1']).sum(),include_groups=False).rename('changed_top1').reset_index()
display(change_summary)


## 5. 결과 저장

제품별 결과는 alpha별로 한 제품당 한 행씩 저장됩니다. 근거 파일에는 어떤 제품 속성으로 어떤 문서를 검색했는지 기록됩니다.


In [ ]:
oof.to_csv('attribute_kb_reranking_oof.csv',index=False,encoding='utf-8-sig')
evidence.to_csv('attribute_kb_reranking_evidence.csv',index=False,encoding='utf-8-sig')
metrics.to_csv('attribute_kb_reranking_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('attribute_kb_reranking_fold_audit.csv',index=False,encoding='utf-8-sig')
print('저장 완료')
